---
### Cómo corre Spring Boot dentro de Colab
Colab no trae Java ni Maven, así que:
1. hay que instalar **JDK 17 + Maven**,
2. escribir el proyecto Spring Boot con `%%writefile`,
3. compilar a un `.jar` y ejecutar.

La **primera compilación descarga las dependencias** (1–3 minutos); las siguientes son rápidas.
Usamos **Spring Boot 3.5.16** (la última de la rama 3.x; esa rama dejó de recibir soporte open source el 30/06/2026, así que para un proyecto nuevo conviene la 4.x). Para este ejercicio alcanza con `spring-boot-starter`: `@Async`, `@EnableAsync` y `ThreadPoolTaskExecutor` vienen en `spring-context`.

Hay que ejecutar las celdas **en orden**. Los tiempos varían según la máquina; lo importante es la proporción.

In [ ]:
%%bash
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y openjdk-17-jdk-headless maven > /dev/null 2>&1
ls /usr/lib/jvm | grep 17
mvn -version | head -1

java-1.17.0-openjdk-amd64
java-17-openjdk-amd64
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [ ]:
import os, glob, subprocess, time
import pandas as pd

# Usamos explícitamente Java 17 (Colab puede traer otra versión por defecto)
os.environ["JAVA_HOME"] = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk-*"))[0]
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

PROY = "/content/spring_async"
JAR  = PROY + "/target/asyncdemo-0.0.1.jar"
os.makedirs(PROY + "/src/main/java/com/ejemplo/asyncdemo", exist_ok=True)

def _texto(p):
    return (p.stdout + p.stderr).decode("utf-8", errors="replace")

def construir():
    """Compila el proyecto con Maven (clean package) y avisa si falla."""
    t0 = time.time()
    p = subprocess.run(["mvn", "-q", "-B", "-ntp", "clean", "package", "-DskipTests"],
                       cwd=PROY, capture_output=True)
    if p.returncode != 0:
        print(_texto(p)[-4000:])
        raise RuntimeError("Falló la compilación con Maven: mirá el error de arriba.")
    print(f"Build OK en {time.time() - t0:.0f} s")

def correr(*args):
    """Ejecuta la app Spring Boot y devuelve todo lo que imprimió."""
    cmd = ["java", "-Dfile.encoding=UTF-8", "-jar", JAR,
           "--spring.main.banner-mode=off", "--logging.level.root=WARN", *args]
    p = subprocess.run(cmd, cwd=PROY, capture_output=True)
    return _texto(p)

print("Java:", os.environ["JAVA_HOME"])

Java: /usr/lib/jvm/java-17-openjdk-amd64


## Proyecto base
`pom.xml` y la clase principal, no cambian durante la actividad.

In [ ]:
%%writefile /content/spring_async/pom.xml
<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
  <modelVersion>4.0.0</modelVersion>

  <parent>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-parent</artifactId>
    <version>3.5.16</version>
    <relativePath/>
  </parent>

  <groupId>com.ejemplo</groupId>
  <artifactId>asyncdemo</artifactId>
  <version>0.0.1</version>

  <properties>
    <java.version>17</java.version>
  </properties>

  <dependencies>
    <!-- Alcanza con el starter básico: @EnableAsync y @Async vienen en spring-context -->
    <dependency>
      <groupId>org.springframework.boot</groupId>
      <artifactId>spring-boot-starter</artifactId>
    </dependency>
  </dependencies>

  <build>
    <plugins>
      <plugin>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-maven-plugin</artifactId>
      </plugin>
    </plugins>
  </build>
</project>

Writing /content/spring_async/pom.xml


In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/AsyncDemoApplication.java
package com.ejemplo.asyncdemo;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class AsyncDemoApplication {

    public static void main(String[] args) {
        // exit(...) cierra el contexto (y con él los pools de threads) para que la JVM termine
        System.exit(SpringApplication.exit(SpringApplication.run(AsyncDemoApplication.class, args)));
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/AsyncDemoApplication.java


---
# Parte A · Versión secuencial
Tres métodos que simulan I/O (esperar a una base de datos o a una API) con `Thread.sleep`: **300, 500 y 400 ms**. Se llaman uno detrás de otro, así que el hilo que llama espera a cada uno antes de seguir.

Se espera un total de **300 + 500 + 400 = 1200 ms**, y que **todo corra en el hilo `main`**.

In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioIO.java
package com.ejemplo.asyncdemo;

import org.springframework.stereotype.Service;

@Service
public class ServicioIO {

    public String consultarUsuario() {
        return simularIO("consultarUsuario", 300, "usuario");
    }

    public String consultarPedidos() {
        return simularIO("consultarPedidos", 500, "pedidos");
    }

    public String consultarFacturas() {
        return simularIO("consultarFacturas", 400, "facturas");
    }

    private String simularIO(String nombre, long ms, String resultado) {
        try {
            Thread.sleep(ms);                       // simula esperar a una base de datos / API
        } catch (InterruptedException e) {
            Thread.currentThread().interrupt();
            throw new IllegalStateException(e);
        }
        System.out.printf("   [%s] %s terminó (%d ms)%n", Thread.currentThread().getName(), nombre, ms);
        return resultado;
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioIO.java


In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/DemoRunner.java
package com.ejemplo.asyncdemo;

import org.springframework.boot.CommandLineRunner;
import org.springframework.stereotype.Component;

@Component
public class DemoRunner implements CommandLineRunner {

    private final ServicioIO servicio;

    public DemoRunner(ServicioIO servicio) {
        this.servicio = servicio;
    }

    @Override
    public void run(String... args) {
        System.out.println("=== PARTE A: secuencial ===");
        System.out.println("Hilo que ejecuta el runner: " + Thread.currentThread().getName());

        long t0 = System.nanoTime();
        String usuario  = servicio.consultarUsuario();    // 300 ms
        String pedidos  = servicio.consultarPedidos();    // 500 ms
        String facturas = servicio.consultarFacturas();   // 400 ms
        long ms = (System.nanoTime() - t0) / 1_000_000;

        System.out.println("Resultados: " + usuario + ", " + pedidos + ", " + facturas);
        System.out.println("TIEMPO TOTAL secuencial: " + ms + " ms  (esperado: aprox. 300 + 500 + 400 = 1200 ms)");
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/DemoRunner.java


In [ ]:
construir()
print(correr())

Build OK en 35 s
=== PARTE A: secuencial ===
Hilo que ejecuta el runner: main
   [main] consultarUsuario terminó (300 ms)
   [main] consultarPedidos terminó (500 ms)
   [main] consultarFacturas terminó (400 ms)
Resultados: usuario, pedidos, facturas
TIEMPO TOTAL secuencial: 1202 ms  (esperado: aprox. 300 + 500 + 400 = 1200 ms)



### Conclusión de la parte A
Sin `@Async`, cada llamada **bloquea** al hilo que la hace: los tiempos se **suman** (≈ 1200 ms) y todo ocurre en `main`. Mientras el hilo espera la respuesta de una "base de datos" no hace nada útil, aunque las tres consultas no dependen entre sí y podrían esperar al mismo tiempo. Esa es la oportunidad que aprovechamos en la parte B.

---
# Parte B · `@Async` con `ThreadPoolTaskExecutor`


Primero la configuración (1 y 2):

In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/AsyncConfig.java
package com.ejemplo.asyncdemo;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.scheduling.annotation.EnableAsync;
import org.springframework.scheduling.concurrent.ThreadPoolTaskExecutor;

@Configuration
@EnableAsync                                   // 1) activa el procesamiento de @Async (crea el proxy)
public class AsyncConfig {

    @Bean(name = "ioExecutor")                 // 2) el pool que va a ejecutar las tareas
    public ThreadPoolTaskExecutor ioExecutor() {
        ThreadPoolTaskExecutor ex = new ThreadPoolTaskExecutor();
        ex.setCorePoolSize(3);                 // threads "fijos"
        ex.setMaxPoolSize(6);                  // tope si la cola se llena
        ex.setQueueCapacity(20);               // tareas en espera
        ex.setThreadNamePrefix("io-");         // así se llaman los threads: io-1, io-2, ...
        return ex;                             // Spring llama a initialize() solo
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/AsyncConfig.java


Ahora `ServicioIO` con `@Async` (3). Los métodos devuelven `CompletableFuture<String>` para que quien llama pueda **esperar el resultado cuando quiera**. Se agrega también un método con una **trampa** muy común (autoinvocación) para verla en acción.

In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioIO.java
package com.ejemplo.asyncdemo;

import java.util.concurrent.CompletableFuture;

import org.springframework.scheduling.annotation.Async;
import org.springframework.stereotype.Service;

@Service
public class ServicioIO {

    @Async("ioExecutor")                       // 3) este método se ejecuta en el pool "ioExecutor"
    public CompletableFuture<String> consultarUsuario() {
        return simularIO("consultarUsuario", 300, "usuario");
    }

    @Async("ioExecutor")
    public CompletableFuture<String> consultarPedidos() {
        return simularIO("consultarPedidos", 500, "pedidos");
    }

    @Async("ioExecutor")
    public CompletableFuture<String> consultarFacturas() {
        return simularIO("consultarFacturas", 400, "facturas");
    }

    // TRAMPA: NO es @Async y llama a métodos @Async de SU PROPIA clase (self-invocation).
    // "this" es el objeto real, no el proxy, así que @Async se ignora y todo corre en el hilo que llama.
    public long autoinvocacion() {
        long t0 = System.nanoTime();
        this.consultarUsuario();
        this.consultarPedidos();
        this.consultarFacturas();
        return (System.nanoTime() - t0) / 1_000_000;
    }

    private CompletableFuture<String> simularIO(String nombre, long ms, String resultado) {
        try {
            Thread.sleep(ms);
        } catch (InterruptedException e) {
            Thread.currentThread().interrupt();
            throw new IllegalStateException(e);
        }
        System.out.printf("   [%s] %s terminó (%d ms)%n", Thread.currentThread().getName(), nombre, ms);
        return CompletableFuture.completedFuture(resultado);
    }
}

Overwriting /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioIO.java


Y el runner, que ahora **lanza las 3 tareas primero y espera después**:

In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/DemoRunner.java
package com.ejemplo.asyncdemo;

import java.util.concurrent.CompletableFuture;

import org.springframework.boot.CommandLineRunner;
import org.springframework.boot.autoconfigure.condition.ConditionalOnProperty;
import org.springframework.stereotype.Component;

@Component
@ConditionalOnProperty(name = "demo.modo", havingValue = "b", matchIfMissing = true)
public class DemoRunner implements CommandLineRunner {

    private final ServicioIO servicio;

    public DemoRunner(ServicioIO servicio) {
        this.servicio = servicio;
    }

    @Override
    public void run(String... args) {
        System.out.println("=== PARTE B: @Async ===");
        System.out.println("Hilo que llama: " + Thread.currentThread().getName());

        long t0 = System.nanoTime();
        CompletableFuture<String> usuario  = servicio.consultarUsuario();   // vuelve al instante
        CompletableFuture<String> pedidos  = servicio.consultarPedidos();   // vuelve al instante
        CompletableFuture<String> facturas = servicio.consultarFacturas();  // vuelve al instante
        long lanzadas = (System.nanoTime() - t0) / 1_000_000;
        System.out.println("Las 3 tareas se lanzaron en " + lanzadas + " ms (el hilo que llama NO esperó nada)");

        System.out.println("El hilo '" + Thread.currentThread().getName()
                + "' espera acá: CompletableFuture.allOf(...).join()");
        CompletableFuture.allOf(usuario, pedidos, facturas).join();         // <- acá se espera el resultado
        long total = (System.nanoTime() - t0) / 1_000_000;

        System.out.println("Resultados: " + usuario.join() + ", " + pedidos.join() + ", " + facturas.join());
        System.out.println("TIEMPO TOTAL con @Async: " + total + " ms  (esperado: aprox. max(300, 500, 400) = 500 ms)");

        System.out.println();
        System.out.println("--- Trampa: autoinvocación (this.metodoAsync()) ---");
        long auto = servicio.autoinvocacion();
        System.out.println("autoinvocacion() tardó " + auto + " ms  <- volvió a ser secuencial, en el hilo que llama");
    }
}

Overwriting /content/spring_async/src/main/java/com/ejemplo/asyncdemo/DemoRunner.java


In [ ]:
construir()
print(correr())

Build OK en 10 s
=== PARTE B: @Async ===
Hilo que llama: main
Las 3 tareas se lanzaron en 12 ms (el hilo que llama NO esperó nada)
El hilo 'main' espera acá: CompletableFuture.allOf(...).join()
   [io-1] consultarUsuario terminó (300 ms)
   [io-3] consultarFacturas terminó (400 ms)
   [io-2] consultarPedidos terminó (500 ms)
Resultados: usuario, pedidos, facturas
TIEMPO TOTAL con @Async: 512 ms  (esperado: aprox. max(300, 500, 400) = 500 ms)

--- Trampa: autoinvocación (this.metodoAsync()) ---
   [main] consultarUsuario terminó (300 ms)
   [main] consultarPedidos terminó (500 ms)
   [main] consultarFacturas terminó (400 ms)
autoinvocacion() tardó 1201 ms  <- volvió a ser secuencial, en el hilo que llama



### Qué pasó por dentro
```
hilo "main" ──► proxy de Spring ──► ThreadPoolTaskExecutor ──► hilo "io-1" ──► consultarUsuario() real
     ▲               │                                       └► hilo "io-2" ──► consultarPedidos() real
     │               │                                       └► hilo "io-3" ──► consultarFacturas() real
     └── CompletableFuture (devuelto al instante, todavía sin resultado)
```
- Al llamar a `servicio.consultarUsuario()` **no se invoca el método real**: se invoca el **proxy**, que le pasa la tarea al pool y devuelve de inmediato un `CompletableFuture` incompleto.
- El método real corre en un thread del pool (`io-N`), y cuando termina **completa** el Future.
- El hilo que llama sigue libre: lanzó las 3 tareas en pocos milisegundos y recién espera en `allOf(...).join()`.

### Conclusión de la parte B
Con `@Async` las tres esperas **se solaparon**: el total pasó de ≈ 1200 ms a ≈ 500 ms (lo que tarda la **más lenta**), y los nombres de thread (`io-1`, `io-2`, `io-3`) muestran que el trabajo lo hizo el **pool**, no `main`. El hilo que llama quedó libre al instante y esperó **solo cuando necesitó los resultados**. También quedó claro que `@Async` funciona gracias a un **proxy**: si se salta (autoinvocación), vuelve a ser secuencial sin ningún error.

---
# Parte C · Prueba de estrés: variar core / max / queue
Mandamos una **ráfaga de 40 solicitudes** (sin pausa entre ellas) a un pool cuyo tamaño configuramos desde la línea de comandos. Cada solicitud tarda 200 ms. Para cada configuración medimos:
- **aceptadas / rechazadas** (¿el pool se llenó?),
- **cola máx** (cuántas tareas llegaron a esperar en la cola) y **pool máx** (cuántos threads se crearon),
- **latencia** de cada solicitud: desde que se envía hasta que termina (**incluye el tiempo en cola**),
- **tiempo de envío**: cuánto tardó el que llama en *enviar* las 40.

Se agregan tres clases: la configuración del pool de estrés, un servicio `@Async` y el runner que dispara la ráfaga.

In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/StressConfig.java
package com.ejemplo.asyncdemo;

import java.util.concurrent.ThreadPoolExecutor;

import org.springframework.beans.factory.annotation.Value;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.scheduling.concurrent.ThreadPoolTaskExecutor;

@Configuration
public class StressConfig {

    // Pool configurable desde la línea de comandos: --estres.core=2 --estres.max=8 --estres.queue=5 ...
    @Bean(name = "estresExecutor")
    public ThreadPoolTaskExecutor estresExecutor(
            @Value("${estres.core:2}") int core,
            @Value("${estres.max:2}") int max,
            @Value("${estres.queue:5}") int queue,
            @Value("${estres.politica:abort}") String politica) {

        ThreadPoolTaskExecutor ex = new ThreadPoolTaskExecutor();
        ex.setCorePoolSize(core);
        ex.setMaxPoolSize(max);
        ex.setQueueCapacity(queue);
        ex.setThreadNamePrefix("estres-");
        // abort      -> si no hay lugar, lanza TaskRejectedException al que llama
        // callerRuns -> si no hay lugar, la tarea la ejecuta el PROPIO hilo que llama (frena al productor)
        ex.setRejectedExecutionHandler("callerRuns".equals(politica)
                ? new ThreadPoolExecutor.CallerRunsPolicy()
                : new ThreadPoolExecutor.AbortPolicy());
        ex.setWaitForTasksToCompleteOnShutdown(true);
        return ex;
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/StressConfig.java


In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioEstres.java
package com.ejemplo.asyncdemo;

import java.util.concurrent.CompletableFuture;

import org.springframework.scheduling.annotation.Async;
import org.springframework.stereotype.Service;

@Service
public class ServicioEstres {

    @Async("estresExecutor")
    public CompletableFuture<String> operacion(int id, long ms) {
        try {
            Thread.sleep(ms);                           // simula una llamada de I/O
        } catch (InterruptedException e) {
            Thread.currentThread().interrupt();
        }
        return CompletableFuture.completedFuture(Thread.currentThread().getName());
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/ServicioEstres.java


In [ ]:
%%writefile /content/spring_async/src/main/java/com/ejemplo/asyncdemo/StressRunner.java
package com.ejemplo.asyncdemo;

import java.util.ArrayList;
import java.util.Collections;
import java.util.List;
import java.util.Set;
import java.util.TreeSet;
import java.util.concurrent.CompletableFuture;
import java.util.concurrent.RejectedExecutionException;
import java.util.concurrent.ThreadPoolExecutor;

import org.springframework.beans.factory.annotation.Qualifier;
import org.springframework.beans.factory.annotation.Value;
import org.springframework.boot.CommandLineRunner;
import org.springframework.boot.autoconfigure.condition.ConditionalOnProperty;
import org.springframework.scheduling.concurrent.ThreadPoolTaskExecutor;
import org.springframework.stereotype.Component;

@Component
@ConditionalOnProperty(name = "demo.modo", havingValue = "estres")
public class StressRunner implements CommandLineRunner {

    private final ServicioEstres servicio;
    private final ThreadPoolTaskExecutor executor;
    private final int solicitudes;
    private final long ms;
    private final int core;
    private final int max;
    private final int queue;
    private final String politica;

    public StressRunner(ServicioEstres servicio,
                        @Qualifier("estresExecutor") ThreadPoolTaskExecutor executor,
                        @Value("${estres.solicitudes:40}") int solicitudes,
                        @Value("${estres.ms:200}") long ms,
                        @Value("${estres.core:2}") int core,
                        @Value("${estres.max:2}") int max,
                        @Value("${estres.queue:5}") int queue,
                        @Value("${estres.politica:abort}") String politica) {
        this.servicio = servicio;
        this.executor = executor;
        this.solicitudes = solicitudes;
        this.ms = ms;
        this.core = core;
        this.max = max;
        this.queue = queue;
        this.politica = politica;
    }

    @Override
    public void run(String... args) {
        List<Long> latencias = Collections.synchronizedList(new ArrayList<>());
        Set<String> hilos = Collections.synchronizedSet(new TreeSet<>());
        List<CompletableFuture<?>> futuros = new ArrayList<>();
        int aceptadas = 0;
        int rechazadas = 0;
        int colaMax = 0;
        int poolMax = 0;

        long t0 = System.nanoTime();
        for (int i = 0; i < solicitudes; i++) {                 // ráfaga: todas "juntas", sin pausa
            final long envio = System.nanoTime();
            try {
                CompletableFuture<String> f = servicio.operacion(i, ms);
                aceptadas++;
                futuros.add(f.whenComplete((hilo, error) -> {
                    latencias.add((System.nanoTime() - envio) / 1_000_000);   // incluye el tiempo en cola
                    if (hilo != null) {
                        hilos.add(hilo);
                    }
                }));
            } catch (RejectedExecutionException e) {            // TaskRejectedException hereda de esta
                rechazadas++;
            }
            ThreadPoolExecutor tpe = executor.getThreadPoolExecutor();
            colaMax = Math.max(colaMax, tpe.getQueue().size());
            poolMax = Math.max(poolMax, tpe.getPoolSize());
        }
        long envioMs = (System.nanoTime() - t0) / 1_000_000;

        CompletableFuture.allOf(futuros.toArray(new CompletableFuture[0])).join();
        long totalMs = (System.nanoTime() - t0) / 1_000_000;

        List<Long> orden = new ArrayList<>(latencias);
        Collections.sort(orden);
        long prom = 0;
        long p95 = 0;
        long maxLat = 0;
        if (!orden.isEmpty()) {
            prom = Math.round(orden.stream().mapToLong(Long::longValue).average().orElse(0));
            p95 = orden.get((int) Math.ceil(0.95 * orden.size()) - 1);
            maxLat = orden.get(orden.size() - 1);
        }

        // Una línea fácil de parsear desde Python:
        System.out.println("RESULTADO|" + core + "|" + max + "|" + queue + "|" + politica + "|" + solicitudes
                + "|" + aceptadas + "|" + rechazadas + "|" + envioMs + "|" + totalMs
                + "|" + prom + "|" + p95 + "|" + maxLat + "|" + colaMax + "|" + poolMax
                + "|" + String.join(",", hilos));
    }
}

Writing /content/spring_async/src/main/java/com/ejemplo/asyncdemo/StressRunner.java


In [ ]:
construir()

Build OK en 9 s


---
# Preguntas de cierre

## 1) ¿Qué entidad ejecutó la tarea?
**Un thread del `ThreadPoolTaskExecutor`** (en la parte B, `io-1`, `io-2`, `io-3`; en la C, `estres-N`), **no el hilo que llamó**. La cadena es: el llamador invoca al **proxy** que Spring creó por `@EnableAsync`; el proxy no ejecuta el método, sino que **lo entrega al executor** indicado en `@Async("ioExecutor")`; y un thread libre del pool lo ejecuta. El código del método es el mismo; cambia *quién* lo corre. Lo confirman los nombres de thread impresos dentro del método.

*Contraejemplos que vimos:* en la parte A y en la **autoinvocación**, la ejecutó `main` (el propio llamador), porque no hubo proxy ni pool de por medio. Y con `callerRuns` y el pool lleno, también la ejecutó `main`.

## 2) ¿Dónde esperó el resultado?
**En el hilo que llamó**, en el punto donde pidió el resultado: `CompletableFuture.allOf(...).join()` (el hilo `main` se bloqueó ahí). Antes de eso **no esperó nada**: lanzó las 3 tareas en pocos milisegundos y quedó libre. Lo importante es que **el método `@Async` no espera a nadie**: solo devuelve un Future que se completa cuando termina. Quien necesita el valor es quien decide cuándo bloquearse con `join()`/`get()`, o si prefiere encadenar con `thenApply`/`thenAccept` para no bloquear. Si el método fuera `void`, **nadie podría esperarlo**.

## 3) ¿Qué pasa si el pool se llena?
Depende de la configuración, y la parte C lo mostró. El pool primero usa los threads `core`, luego la **cola**, luego crea threads hasta `max`, y **recién entonces rechaza**:
- **`abort` (por defecto):** el llamador recibe `TaskRejectedException` (se pierde la solicitud si no se maneja).
- **`callerRuns`:** la tarea la ejecuta el hilo que llama, que se **frena** (backpressure).
- **Cola grande:** no hay rechazo, pero la **latencia crece** y la memoria también.

Antes de llegar a rechazar, ya se nota en la **latencia**: las tareas esperan en la cola. Por eso conviene monitorear el tamaño de la cola y el tiempo de espera, no solo los errores.

---
# Conclusión final del ejercicio

**`@Async` le pasa el trabajo a otro thread y devuelve un "ticket" (el Future) para recoger el resultado después.** En la parte A todo se hizo en fila: los tiempos se sumaban (≈ 1200 ms) y el hilo principal esperaba sin hacer nada. En la B, con `@EnableAsync`, un pool y `@Async`, las tres esperas se solaparon: el total bajó a lo que tarda la tarea más lenta (≈ 500 ms), el trabajo lo hicieron threads del pool (`io-1`, `io-2`, `io-3`) y quien llamó siguió libre hasta que necesitó el resultado.

Pero no es magia ni es gratis. `@Async` funciona por un **proxy**: si se lo saltea (autoinvocación) o se olvida `@EnableAsync`, vuelve a ser secuencial **sin avisar**. Y el pool tiene **límites**: en la parte C vimos que cuando llega más trabajo del que se puede atender hay que decidir qué falla primero. Se pueden rechazar solicitudes (rápido, pero se pierden), acumularlas en una cola (nada falla, pero sube la latencia y la memoria) o frenar a quien las envía (`callerRuns`).

**`@Async` sirve para tareas de espera (I/O) independientes entre sí; hay que darle un pool con nombre y con tamaño pensado, devolver un `CompletableFuture` para poder esperar o manejar errores, y probarlo bajo carga mirando cola, rechazos y latencia**.